In [5]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "energyiq").is_dir():
    if root.parent == root:
        raise RuntimeError("Couldn't find the energyiq package above this notebook")
    root = root.parent

sys.path.insert(0, str(root))
print("Importing from", root)

Importing from c:\Users\benwa\OneDrive\Desktop\EnergyIQ\backend


In [6]:
import pandas as pd
from sqlalchemy.orm import Session, DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker
from sqlalchemy import create_engine, func, select, text

from energyiq.models import Building, MeterReading, Weather 
from energyiq import config 

pd.set_option("display.max_columns", None)   # show every column
pd.set_option("display.width", None)         # don't wrap to a fixed width

building_name = "Ohio Stadium"
db_path = config.DB_PATH
db_path.parent.mkdir(parents=True, exist_ok=True)
engine = create_engine(f"duckdb:///{db_path}")
SessionLocal = sessionmaker(bind=engine)

with SessionLocal.begin() as session:
  result = session.execute(
      text("SELECT id, sitename FROM buildings WHERE sitename =:name"),
      {"name": building_name}
  )
  bId = result.first()[0]

  df = pd.read_sql(text("""
      SELECT m.*, b.sitename, w.*
      FROM meter_readings m
      JOIN buildings b ON m.building_id = b.id
      JOIN weather w
        ON date_trunc('hour', m.readingtime) = date_trunc('hour', w.date)
      WHERE m.building_id = :bId
  """), session.connection(), params={"bId": bId})


In [7]:
def drop_outliers(df, col="readingvalue", group="meterid", k=10):
    df = df[df[col] >= 0]   # negative usage is impossible

    med = df.groupby(group)[col].transform("median")
    mad = df.groupby(group)[col].transform(lambda s: (s - s.median()).abs().median())
    mad = mad.replace(0, df[col].abs().median())   # avoid dividing by zero for flat meters

    keep = (df[col] - med).abs() <= k * mad
    print(f"dropping {(~keep).sum()} of {len(df)} rows")
    return df[keep]

df = drop_outliers(df)

print(df.groupby("meterid")["readingvalue"].describe())

dropping 7052 of 552981 rows
           count       mean        std       min        25%        50%  \
meterid                                                                  
246213   52503.0  17.747677   2.305282  4.201364  16.258515  17.453432   
246214   55113.0  29.174060  11.612855  0.000000  22.984388  26.094608   
246215   55290.0   7.107794   7.397908  0.000000   0.000000   2.293523   
246216   55297.0  55.718764  26.485363  0.000000  34.163269  57.476318   
246217   55112.0  40.134458  14.532593  0.000000  30.649514  38.814646   
246218   54813.0  36.078153  16.346112  0.000000  23.616628  26.626117   
246219   55237.0  51.800852  13.777456  0.000000  42.127337  47.653902   
246220   55218.0  52.744277  13.836444  0.000000  43.675019  50.759679   
246221   55134.0  89.822997  25.722916  0.000000  70.222916  80.276637   
247516   52212.0  83.104107  98.866429  0.000000  12.384731  31.400239   

                75%         max  
meterid                          
246213    18.7

In [ ]:
df = df.sort_values(["meterid", "readingtime"])
g = df.groupby("meterid")["readingvalue"]

df["lag_24h"] = g.shift(96)     # 96 × 15 min = 24 hours
df["lag_7d"]  = g.shift(672)    # 672 × 15 min = 7 days

df = df.dropna(subset=["lag_24h", "lag_7d"])

In [8]:
df = pd.get_dummies(df, columns=["utility", "meterid"], dtype=int)
df["is_weekend"] = (df["readingtime"].dt.dayofweek >= 5).astype(int)

def time_bucket(hour):
    if hour >= 22 or hour < 6:
        return "night"
    if hour < 9:
        return "early_morning"
    if hour < 12:
        return "late_morning"
    if hour < 15:
        return "early_afternoon"
    if hour < 19:
        return "late_afternoon"
    return "evening"

df["time_of_day"] = df["readingtime"].dt.hour.map(time_bucket)
df = pd.get_dummies(df, columns=["time_of_day"], prefix="tod", dtype=int)

dates = df["readingtime"].dt.normalize()     # drop the time, so the end day counts in full

breaks = [
    ("10/12/23", "10/13/23"), 
    ("11/22/23", "11/24/23"),
    ("12/15/23", "1/7/24"),
    ("3/11/24", "3/15/24"),
    ("5/1/24", "8/19/24"),
    ("10/10/24", "10/11/24"), 
    ("11/27/24", "11/29/24"),
    ("12/13/24", "1/5/25"), 
    ("3/10/25", "3/14/25"),
    ("4/30/25", "8/18/25"),
    ("10/16/25", "10/17/25"), 
    ("11/26/25", "11/28/25"),
    ("12/19/25", "1/11/26"),
    ("3/16/26", "3/20/26"),
    ("5/5/26", "8/24/26"),
    ("10/15/26", "10/16/26"),
    ("11/25/26", "11/27/26"),
    ("11/18/26", "1/10/27"), 
    ("3/15/27", "3/19/27"),
    ("5/5/27", "8/23/27")
]

on_break = pd.Series(False, index=df.index)
for start, end in breaks:
    s = pd.to_datetime(start, format="%m/%d/%y")
    e = pd.to_datetime(end, format="%m/%d/%y")
    on_break |= dates.between(s, e)          # inclusive on both ends

df["in_classes"] = (~on_break).astype(int)
df = df.drop(columns=["readingunits", "sitename", "latitude", "longitude", "building_id", "date"], errors="ignore")

df = df.sort_values("readingtime").set_index("readingtime")
X = df.drop(columns=["readingvalue"])
Y = df["readingvalue"]
X.head()

,temperature_2m,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance,relative_humidity_2m,dew_point_2m,precipitation,wind_speed_10m,wind_speed_100m,wind_direction_100m,wind_direction_10m,cloud_cover,apparent_temperature,utility_ELECTRICITY,utility_HEAT,meterid_246213,meterid_246214,meterid_246215,meterid_246216,meterid_246217,meterid_246218,meterid_246219,meterid_246220,meterid_246221,meterid_247516,is_weekend,tod_early_afternoon,tod_early_morning,tod_evening,tod_late_afternoon,tod_late_morning,tod_night,in_classes
readingtime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2024-09-01 05:00:00,69.999802,0.0,0.0,0.0,0.0,94.023643,68.199799,0.0,3.414632,7.765323,258.366394,238.392548,88.0,75.615707,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1
2024-09-01 05:00:00,69.999802,0.0,0.0,0.0,0.0,94.023643,68.199799,0.0,3.414632,7.765323,258.366394,238.392548,88.0,75.615707,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1
2024-09-01 05:00:00,69.999802,0.0,0.0,0.0,0.0,94.023643,68.199799,0.0,3.414632,7.765323,258.366394,238.392548,88.0,75.615707,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1
2024-09-01 05:00:00,69.999802,0.0,0.0,0.0,0.0,94.023643,68.199799,0.0,3.414632,7.765323,258.366394,238.392548,88.0,75.615707,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1
2024-09-01 05:00:00,69.999802,0.0,0.0,0.0,0.0,94.023643,68.199799,0.0,3.414632,7.765323,258.366394,238.392548,88.0,75.615707,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,1
